# catboost — r2_full_causal
Independent validation-screen run; seed 42. Validation selects class weighting/calibration and threshold policy. Final 40% is prediction-only.

In [ ]:
import base64, gzip, json, sys
from pathlib import Path
bundle=json.loads(gzip.decompress(base64.b64decode("...")))
pkg=Path("/kaggle/working/fraudlab_pkg/fraudlab"); pkg.mkdir(parents=True,exist_ok=True)
for name,source in bundle.items(): (pkg/name).write_text(source,encoding="utf-8")
sys.path.insert(0,str(pkg.parent))
print("Project modules loaded",len(bundle))


In [ ]:
from pathlib import Path
import json, subprocess, time
import catboost
from catboost.utils import get_gpu_device_count
from fraudlab.pipeline import train_recipe_model_from_artifacts
from fraudlab.config import ExperimentConfig, BanditConfig
MODEL_NAME="catboost"
RECIPE="r2_full_causal"
SEED=42
gpu_count=get_gpu_device_count()
if gpu_count < 1:
    raise RuntimeError("Kaggle GPU is not enabled or no CatBoost-compatible GPU is visible; refusing CPU fallback.")
gpu_name=subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],capture_output=True,text=True,check=True).stdout.strip().splitlines()[0]
print(f"CatBoost {catboost.__version__} | GPU devices={gpu_count} | GPU={gpu_name} | seed={SEED}")
root=Path("/kaggle/input")
files=list(root.rglob("ieee_cis_features.parquet"))
tx=[p for p in files if p.parent.name=="transaction"]
if len(tx) != 1: raise FileNotFoundError(f"Expected exactly one transaction feature artifact from preprocessing v12; found {len(tx)}")
print(f"Transaction features: {tx[0]}")
config=ExperimentConfig(random_seed=SEED,bandit=BanditConfig(seed=SEED),catboost_task_type="GPU")
out=Path("/kaggle/working/outputs/model_grid")/(MODEL_NAME+"__"+RECIPE+"__seed_"+str(SEED)+"__gpu")
started=time.time()
result=train_recipe_model_from_artifacts(tx[0],None,RECIPE,MODEL_NAME,out,config)
record={"model":MODEL_NAME,"recipe":RECIPE,"seed":SEED,"compute":"GPU","gpu_name":gpu_name,"catboost_version":catboost.__version__,"runtime_seconds":time.time()-started,"result":result}
(out/"run_record.json").write_text(json.dumps(record,indent=2,default=str),encoding="utf-8")
print(json.dumps({"model":MODEL_NAME,"recipe":RECIPE,"compute":"GPU","gpu_name":gpu_name,"runtime_seconds":round(time.time()-started,1),"validation":result["validation_metrics"],"validation_cost":result["validation_total_cost"]},indent=2))